## Dataset Testing Notebooks

This notebook is for testing and understanding the setup and organization of the sequence data from MPC sim.

This set will also be used to take the sequence data, clean it, and convert it to tensors or another format for use with pytorch training

In [50]:
#imports 
import numpy as np
import os 

## Section 1: Loading a single datset and inspecting data 

In [51]:
#set home 
from pathlib import Path

# Get the project root (parent of notebooks directory)
project_root = Path.cwd().parent
print(f"Project root: {project_root}")

#set paths for single map, spielberg
single_map_path = project_root / "mpc_datasets" / "run_20260409T193833Z" / "Spielberg" / "normal" / "kmpc_Spielberg_normal.npz"
print(f"Single map path: {single_map_path}")

#unzip .npz option
data = np.load(single_map_path)

Project root: /home/devin_work/work/f1tenth/ApproxiMPC
Single map path: /home/devin_work/work/f1tenth/ApproxiMPC/mpc_datasets/run_20260409T193833Z/Spielberg/normal/kmpc_Spielberg_normal.npz


In [52]:
#view and organize different arrays 
print(data.files)

for key in data.files:
    print(f"Key: {key}, Shape: {data[key].shape}, Dtype: {data[key].dtype}")


['observations', 'next_observations', 'expert_actions', 'executed_actions', 'rewards', 'terminations', 'truncations', 'is_perturbed', 'collector_lap_counts', 'env_lap_counts', 'collision_flags', 'boundary_flags', 'episode_ids', 'step_ids', 'feature_names', 'action_names', 'noise_vectors', 'lidar_scans', 'next_lidar_scans', 'lidar_names']
Key: observations, Shape: (87428, 5), Dtype: float32
Key: next_observations, Shape: (87428, 5), Dtype: float32
Key: expert_actions, Shape: (87428, 2), Dtype: float32
Key: executed_actions, Shape: (87428, 2), Dtype: float32
Key: rewards, Shape: (87428,), Dtype: float32
Key: terminations, Shape: (87428,), Dtype: bool
Key: truncations, Shape: (87428,), Dtype: bool
Key: is_perturbed, Shape: (87428,), Dtype: bool
Key: collector_lap_counts, Shape: (87428,), Dtype: int32
Key: env_lap_counts, Shape: (87428,), Dtype: int32
Key: collision_flags, Shape: (87428,), Dtype: bool
Key: boundary_flags, Shape: (87428,), Dtype: bool
Key: episode_ids, Shape: (87428,), Dtyp

In [53]:
#inspect individual data types
print("observations names")
print(data["feature_names"])
print("action names")
print(data["action_names"])
# print(data["expert_actions"])
# print(data["executed_actions"])
print("lidar")
print(data["lidar_names"])
#only 60 lidar scans? double check
# print(data["lidar_scans"])


observations names
['pose_x' 'pose_y' 'delta' 'linear_vel_x' 'pose_theta']
action names
['steering_angle' 'speed']
lidar
['lidar_0' 'lidar_1' 'lidar_2' 'lidar_3' 'lidar_4' 'lidar_5' 'lidar_6'
 'lidar_7' 'lidar_8' 'lidar_9' 'lidar_10' 'lidar_11' 'lidar_12' 'lidar_13'
 'lidar_14' 'lidar_15' 'lidar_16' 'lidar_17' 'lidar_18' 'lidar_19'
 'lidar_20' 'lidar_21' 'lidar_22' 'lidar_23' 'lidar_24' 'lidar_25'
 'lidar_26' 'lidar_27' 'lidar_28' 'lidar_29' 'lidar_30' 'lidar_31'
 'lidar_32' 'lidar_33' 'lidar_34' 'lidar_35' 'lidar_36' 'lidar_37'
 'lidar_38' 'lidar_39' 'lidar_40' 'lidar_41' 'lidar_42' 'lidar_43'
 'lidar_44' 'lidar_45' 'lidar_46' 'lidar_47' 'lidar_48' 'lidar_49'
 'lidar_50' 'lidar_51' 'lidar_52' 'lidar_53' 'lidar_54' 'lidar_55'
 'lidar_56' 'lidar_57' 'lidar_58' 'lidar_59']


## What columns and data to keep

### Inputs
1. observations: pose (x,y,theta), yaw, and linear velocity. Drop x and y coordinates. Keep theta (current steering angle), yaw angle, and current linear velocity
3. lidar_scans: current timestep scans, check with henry about number of rays 
3. episode_id: useful for splitting up train, val, and test
8. step_ids: not used as input, used to organize overall rows (for human readability)

### Outputs
1. expert_actions: desired output for the current timestep. speed and steering angle

### Drop, make note of them

9. feature_names: good to keep, helpful for reading data, never passed as inputs
10. action_names: good to keep, helpful for reading data, never passed as inputs
11. lidar_names: probably not necessary. But could be good to have 

### Drop
1. next observations: the next timestep of observations, unnecessary
2. executed_actions: desired action of the agent, before the execution DOUBLE CHECK (includes random noise)
3. rewards: rewards for MPC not useful for LSTM
4. is_perturbed: bool for MPC input, not needed for LSTM
5. next_lidar_scans: next timstep of lidar, not available during inferenc

Additional flags for MPC, should be unnecessary
1. terminations: check
2. truncations: check
3. collector_lap_counts: cehck
4. env_lap_counts: check
5. collision_flag: check
6. boundary_flags: check
7. noise_vectors: check




## Section 2: Data Cleaning and Prep Test

Drop the non needed columns

Save existing arrays, and make edits as needed (drop x and y)

Concatonate the arrays to a final object

Convert to a tensor

In [57]:
#info cols (for splits)
map_name = np.full(len(data["step_ids"]), "Spielberg_normal")
episodes = data["episode_ids"]
steps = data["step_ids"]

#input cols
observations = data["observations"]
lidar_scans = data["lidar_scans"]

#output col
expert_actions = data["expert_actions"]


In [58]:
#clean up observations 
observations = observations[:, 2:]
print(observations)

[[ 0.0000000e+00  0.0000000e+00  3.4042091e+00]
 [ 0.0000000e+00  3.8039997e-02  3.4042091e+00]
 [ 0.0000000e+00  1.3314000e-01  3.4042091e+00]
 ...
 [-3.2000002e-02  4.0108986e+00  3.4072411e+00]
 [ 4.1633363e-17  4.0109091e+00  3.4052973e+00]
 [-3.2000002e-02  4.0109239e+00  3.4033535e+00]]


In [ ]:
#concatenate input features (observations + lidar) along axis=1
# observations shape: (N, 3) -> [delta, linear_vel_x, pose_theta]
# lidar_scans shape: (N, 60) -> lidar measurements
# inputs = np.concatenate([observations, lidar_scans], axis=1)
# print(f"inputs shape: {inputs.shape}")  # Should be (N, 63)
# print(f"expert_actions shape: {expert_actions.shape}")  # Should be (N, 2)

# # Metadata stays separate for train/val/test splitting
# print(f"\nMetadata:")
# print(f"map_name: {map_name.shape}")
# print(f"episodes: {episodes.shape}")
# print(f"steps: {steps.shape}")

TypeError: concatenate() takes from 1 to 3 positional arguments but 6 were given

In [ ]:
#convery to pytorch tensor!
#single tensor for each episode

#then assemble the tensor into a tensor of tensors 

## Section 3: Inspect Data 